# Práctica — Clase 5: Aprendizaje estructurado y adversarial

**Objetivos de esta práctica**

- Comprobar de forma experimental el efecto del hiperparámetro `adv_multiplier` en la robustez de un clasificador de imágenes frente a ejemplos perturbados
- Comprobar el efecto del multiplicador de regularización de grafos sobre la precisión de un modelo entrenado con un grafo natural (Cora)

Completa las celdas marcadas con `# TODO`. Cuando termines cada ejercicio, despliega la solución para comparar tu enfoque, pero intenta resolverlo por tu cuenta primero.


In [ ]:
# Instala Neural Structured Learning (NSL) si no está disponible
!pip install --quiet neural-structured-learning


In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import neural_structured_learning as nsl

print('TensorFlow:', tf.__version__)


## Ejercicio 1 — El efecto de `adv_multiplier`

En el notebook de teoría viste que un modelo con regularización adversarial mantiene mucha más precisión frente a imágenes perturbadas que un modelo base. Esa mejora depende del hiperparámetro `adv_multiplier`: el peso que se le da al término adversarial dentro de la pérdida total.

**Tu tarea:** completa la función `train_and_evaluate_adv_model` para que, dado un valor de `adv_multiplier`, entrene un modelo con `AdversarialRegularization` sobre MNIST y devuelva su precisión (accuracy) tanto sobre imágenes limpias como sobre imágenes perturbadas. Después, ejecuta la comparación con al menos tres valores distintos de `adv_multiplier` (por ejemplo, `0.0`, `0.2` y `0.8`) y observa qué ocurre.

**Pistas:**
- `adv_multiplier=0.0` equivale, en la práctica, a no aplicar regularización adversarial: es tu referencia (el modelo base)
- Para generar los ejemplos perturbados sobre los que evalúas, usa el propio modelo entrenado (`adv_model.perturb_on_batch(...)`), tal y como se hizo en el notebook de teoría
- Un `adv_multiplier` más alto suele mejorar la accuracy sobre ejemplos perturbados, pero puede reducir ligeramente la accuracy sobre datos limpios: ese equilibrio (trade-off) es justo lo que estás explorando


In [ ]:
# --- Preparación de datos (ya resuelta, no hace falta tocarla) ---
IMAGE_INPUT_NAME = 'image'
LABEL_INPUT_NAME = 'label'

datasets = tfds.load('mnist')
train_dataset_adv = datasets['train']
test_dataset_adv = datasets['test']

def normalize(features):
    features[IMAGE_INPUT_NAME] = tf.cast(features[IMAGE_INPUT_NAME], tf.float32) / 255.0
    return features

def convert_to_dictionaries(features):
    return {IMAGE_INPUT_NAME: features[IMAGE_INPUT_NAME], LABEL_INPUT_NAME: features[LABEL_INPUT_NAME]}

BATCH_SIZE = 32
train_set_for_adv_model = train_dataset_adv.map(normalize).shuffle(10000).batch(BATCH_SIZE).map(convert_to_dictionaries)
test_set_for_adv_model = test_dataset_adv.map(normalize).batch(BATCH_SIZE).map(convert_to_dictionaries)


def build_base_model():
    """CNN sencilla: 3 capas convolucionales + 2 densas."""
    inputs = tf.keras.Input(shape=(28, 28, 1), dtype=tf.float32, name=IMAGE_INPUT_NAME)
    x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(10)(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs)


In [ ]:
def train_and_evaluate_adv_model(adv_multiplier, epochs=3, adv_step_size=0.2):
    """Entrena un modelo con AdversarialRegularization para un adv_multiplier dado.

    Devuelve una tupla (accuracy_datos_limpios, accuracy_datos_perturbados).
    """
    # TODO: crea la configuración adversarial con nsl.configs.make_adv_reg_config
    # usando adv_multiplier, adv_step_size y adv_grad_norm='infinity'
    adv_config = None

    # TODO: construye el modelo base con build_base_model() y envuélvelo con
    # nsl.keras.AdversarialRegularization (recuerda pasar label_keys=[LABEL_INPUT_NAME])
    adv_model = None

    # TODO: compila el modelo (optimizer='adam', loss SparseCategoricalCrossentropy
    # from_logits=True, metrics=['acc']) y entrénalo con train_set_for_adv_model
    # durante `epochs` épocas

    # Evaluamos sobre datos limpios
    clean_results = dict(zip(adv_model.metrics_names,
                              adv_model.evaluate(test_set_for_adv_model, verbose=0)))
    clean_accuracy = clean_results['sparse_categorical_accuracy']

    # Generamos ejemplos perturbados a partir de este mismo modelo y medimos su accuracy
    perturbed_metric = tf.keras.metrics.SparseCategoricalAccuracy()
    for batch in test_set_for_adv_model:
        perturbed_batch = adv_model.perturb_on_batch(batch)
        perturbed_batch[IMAGE_INPUT_NAME] = tf.clip_by_value(
            perturbed_batch[IMAGE_INPUT_NAME], 0.0, 1.0)
        y_true = perturbed_batch.pop(LABEL_INPUT_NAME)
        y_pred = adv_model.base_model(perturbed_batch)
        perturbed_metric(y_true, y_pred)

    return clean_accuracy, perturbed_metric.result().numpy()


# TODO: ejecuta la comparación con al menos tres valores de adv_multiplier
# (por ejemplo 0.0, 0.2 y 0.8) y muestra la accuracy limpia y perturbada de cada uno
multiplier_values = []  # TODO: rellena la lista de valores a comparar

for multiplier in multiplier_values:
    clean_acc, perturbed_acc = train_and_evaluate_adv_model(multiplier)
    print(f'adv_multiplier={multiplier}: accuracy limpia={clean_acc:.3f}, accuracy perturbada={perturbed_acc:.3f}')


<details>
<summary>💡 Ver solución</summary>

```python
def train_and_evaluate_adv_model(adv_multiplier, epochs=3, adv_step_size=0.2):
    """Entrena un modelo con AdversarialRegularization para un adv_multiplier dado.

    Devuelve una tupla (accuracy_datos_limpios, accuracy_datos_perturbados).
    """
    # Configuración adversarial: multiplier controla el peso del término
    # adversarial en la pérdida total; adv_step_size, la magnitud de la perturbación
    adv_config = nsl.configs.make_adv_reg_config(
        multiplier=adv_multiplier,
        adv_step_size=adv_step_size,
        adv_grad_norm='infinity')

    # Construimos el modelo base y lo envolvemos con AdversarialRegularization.
    # label_keys indica qué feature del diccionario contiene la etiqueta, porque
    # el modelo adversarial necesita la etiqueta para calcular la perturbación.
    base_model = build_base_model()
    adv_model = nsl.keras.AdversarialRegularization(
        base_model, label_keys=[LABEL_INPUT_NAME], adv_config=adv_config)

    adv_model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['acc'])
    adv_model.fit(train_set_for_adv_model, epochs=epochs, verbose=0)

    # Evaluamos sobre datos limpios
    clean_results = dict(zip(adv_model.metrics_names,
                              adv_model.evaluate(test_set_for_adv_model, verbose=0)))
    clean_accuracy = clean_results['sparse_categorical_accuracy']

    # Generamos ejemplos perturbados a partir de este mismo modelo y medimos su accuracy
    perturbed_metric = tf.keras.metrics.SparseCategoricalAccuracy()
    for batch in test_set_for_adv_model:
        perturbed_batch = adv_model.perturb_on_batch(batch)
        perturbed_batch[IMAGE_INPUT_NAME] = tf.clip_by_value(
            perturbed_batch[IMAGE_INPUT_NAME], 0.0, 1.0)
        y_true = perturbed_batch.pop(LABEL_INPUT_NAME)
        y_pred = adv_model.base_model(perturbed_batch)
        perturbed_metric(y_true, y_pred)

    return clean_accuracy, perturbed_metric.result().numpy()


# adv_multiplier=0.0 equivale a no aplicar regularización adversarial: es la referencia
multiplier_values = [0.0, 0.2, 0.8]

for multiplier in multiplier_values:
    clean_acc, perturbed_acc = train_and_evaluate_adv_model(multiplier)
    print(f'adv_multiplier={multiplier}: accuracy limpia={clean_acc:.3f}, accuracy perturbada={perturbed_acc:.3f}')

# Resultado esperado (orden de magnitud, puede variar entre ejecuciones):
# adv_multiplier=0.0 -> accuracy limpia alta, accuracy perturbada muy baja (~0.5 o menos)
# adv_multiplier=0.2 -> accuracy limpia similar, accuracy perturbada mucho más alta (~0.9)
# adv_multiplier=0.8 -> accuracy perturbada aún mejor, accuracy limpia puede bajar un poco
```
</details>


## Ejercicio 2 — El efecto del multiplicador de regularización de grafos

Ahora vuelve al dataset Cora (grafo natural de citas entre artículos científicos) que viste en el notebook de teoría. El parámetro `graph_regularization_multiplier` controla el peso del término de regularización de grafos en la pérdida total, de forma parecida a como `adv_multiplier` controlaba el peso del término adversarial en el ejercicio anterior.

**Tu tarea:** completa la función `train_and_evaluate_graph_model` para que, dado un valor de `graph_regularization_multiplier`, entrene un MLP envuelto con `GraphRegularization` sobre Cora y devuelva su accuracy en el test set. Después, compara al menos tres valores (por ejemplo, `0.01`, `0.1` y `1.0`).

**Pistas:**
- Un multiplicador demasiado bajo apenas se diferencia del modelo base (sin regularización de grafos)
- Un multiplicador demasiado alto puede hacer que el modelo priorice tanto "parecerse a sus vecinos" que aprenda peor la tarea de clasificación en sí
- Reutiliza `make_mlp_functional_model` y `make_dataset`, que ya están preparadas en la celda de configuración


In [ ]:
# --- Descarga y preprocesado de Cora (ya resuelto, no hace falta tocarlo) ---
!wget --quiet -P /tmp https://linqs-data.soe.ucsc.edu/public/lbc/cora.tgz
!tar -C /tmp -xzf /tmp/cora.tgz
!wget --quiet https://raw.githubusercontent.com/tensorflow/neural-structured-learning/master/neural_structured_learning/examples/preprocess/cora/preprocess_cora_dataset.py

!python preprocess_cora_dataset.py \
  --input_cora_content=/tmp/cora/cora.content \
  --input_cora_graph=/tmp/cora/cora.cites \
  --max_nbrs=5 \
  --output_train_data=/tmp/cora/train_merged_examples.tfr \
  --output_test_data=/tmp/cora/test_examples.tfr

NBR_FEATURE_PREFIX = 'NL_nbr_'
NBR_WEIGHT_SUFFIX = '_weight'
MAX_SEQ_LENGTH = 1433
NUM_CLASSES = 7


def make_dataset(file_path, num_neighbors, batch_size=128, training=False):
    def parse_example(example_proto):
        feature_spec = {
            'words': tf.io.FixedLenFeature(
                [MAX_SEQ_LENGTH], tf.int64,
                default_value=tf.constant(0, dtype=tf.int64, shape=[MAX_SEQ_LENGTH])),
            'label': tf.io.FixedLenFeature((), tf.int64, default_value=-1),
        }
        if training:
            for i in range(num_neighbors):
                nbr_feature_key = '{}{}_{}'.format(NBR_FEATURE_PREFIX, i, 'words')
                nbr_weight_key = '{}{}{}'.format(NBR_FEATURE_PREFIX, i, NBR_WEIGHT_SUFFIX)
                feature_spec[nbr_feature_key] = tf.io.FixedLenFeature(
                    [MAX_SEQ_LENGTH], tf.int64,
                    default_value=tf.constant(0, dtype=tf.int64, shape=[MAX_SEQ_LENGTH]))
                feature_spec[nbr_weight_key] = tf.io.FixedLenFeature(
                    [1], tf.float32, default_value=tf.constant([0.0]))
        features = tf.io.parse_single_example(example_proto, feature_spec)
        label = features.pop('label')
        return features, label

    dataset = tf.data.TFRecordDataset([file_path])
    if training:
        dataset = dataset.shuffle(10000)
    dataset = dataset.map(parse_example)
    dataset = dataset.batch(batch_size)
    return dataset


def make_mlp_functional_model(num_fc_units=(50, 50), dropout_rate=0.5):
    inputs = tf.keras.Input(shape=(MAX_SEQ_LENGTH,), dtype='int64', name='words')
    cur_layer = tf.keras.layers.Lambda(lambda x: tf.keras.backend.cast(x, tf.float32))(inputs)
    for num_units in num_fc_units:
        cur_layer = tf.keras.layers.Dense(num_units, activation='relu')(cur_layer)
        cur_layer = tf.keras.layers.Dropout(dropout_rate)(cur_layer)
    outputs = tf.keras.layers.Dense(NUM_CLASSES)(cur_layer)
    return tf.keras.Model(inputs, outputs=outputs)


In [ ]:
def train_and_evaluate_graph_model(graph_regularization_multiplier, num_neighbors=1, train_epochs=15):
    """Entrena un MLP + GraphRegularization sobre Cora para un multiplicador dado.

    Devuelve la accuracy en el test set.
    """
    train_dataset = make_dataset('/tmp/cora/train_merged_examples.tfr',
                                  num_neighbors=num_neighbors, training=True)
    test_dataset = make_dataset('/tmp/cora/test_examples.tfr', num_neighbors=num_neighbors)

    # TODO: crea graph_reg_config con nsl.configs.make_graph_reg_config, usando
    # max_neighbors=num_neighbors, multiplier=graph_regularization_multiplier,
    # distance_type=nsl.configs.DistanceType.L2 y sum_over_axis=-1
    graph_reg_config = None

    # TODO: construye el modelo base con make_mlp_functional_model() y envuélvelo
    # con nsl.keras.GraphRegularization usando graph_reg_config
    graph_model = None

    # TODO: compila el modelo (optimizer='adam', loss SparseCategoricalCrossentropy
    # from_logits=True, metrics=['accuracy']) y entrénalo con train_dataset
    # durante `train_epochs` épocas

    eval_results = dict(zip(graph_model.metrics_names,
                             graph_model.evaluate(test_dataset, verbose=0)))
    return eval_results['accuracy']


# TODO: compara al menos tres valores de graph_regularization_multiplier
# (por ejemplo 0.01, 0.1 y 1.0) y muestra la accuracy de cada uno
multiplier_values = []  # TODO: rellena la lista de valores a comparar

for multiplier in multiplier_values:
    acc = train_and_evaluate_graph_model(multiplier)
    print(f'graph_regularization_multiplier={multiplier}: accuracy={acc:.3f}')


<details>
<summary>💡 Ver solución</summary>

```python
def train_and_evaluate_graph_model(graph_regularization_multiplier, num_neighbors=1, train_epochs=15):
    """Entrena un MLP + GraphRegularization sobre Cora para un multiplicador dado.

    Devuelve la accuracy en el test set.
    """
    train_dataset = make_dataset('/tmp/cora/train_merged_examples.tfr',
                                  num_neighbors=num_neighbors, training=True)
    test_dataset = make_dataset('/tmp/cora/test_examples.tfr', num_neighbors=num_neighbors)

    # graph_reg_config controla cuántos vecinos se usan y el peso del término
    # de regularización de grafos frente a la pérdida de clasificación estándar
    graph_reg_config = nsl.configs.make_graph_reg_config(
        max_neighbors=num_neighbors,
        multiplier=graph_regularization_multiplier,
        distance_type=nsl.configs.DistanceType.L2,
        sum_over_axis=-1)

    # Envolvemos un MLP base (sin ningún cambio de arquitectura) con GraphRegularization
    base_model = make_mlp_functional_model()
    graph_model = nsl.keras.GraphRegularization(base_model, graph_reg_config)

    graph_model.compile(
        optimizer='adam',
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'])
    graph_model.fit(train_dataset, epochs=train_epochs, verbose=0)

    eval_results = dict(zip(graph_model.metrics_names,
                             graph_model.evaluate(test_dataset, verbose=0)))
    return eval_results['accuracy']


multiplier_values = [0.01, 0.1, 1.0]

for multiplier in multiplier_values:
    acc = train_and_evaluate_graph_model(multiplier)
    print(f'graph_regularization_multiplier={multiplier}: accuracy={acc:.3f}')

# Resultado esperado (orden de magnitud, puede variar entre ejecuciones):
# multiplier=0.01 -> accuracy parecida a la del modelo base, sin apenas efecto del grafo
# multiplier=0.1  -> mejor accuracy: el valor usado en el tutorial oficial de NSL
# multiplier=1.0  -> el término de grafo domina la pérdida; la accuracy puede
#                    estabilizarse o incluso empeorar si el modelo deja de aprender
#                    bien la tarea de clasificación en sí
```
</details>


## Para terminar

Los dos ejercicios comparten la misma idea de fondo: un multiplicador (`adv_multiplier` o `graph_regularization_multiplier`) controla cuánto peso le das, dentro de la pérdida total, a la señal adicional que aporta NSL (perturbación adversarial o estructura de grafo) frente a la pérdida de clasificación estándar. Encontrar un buen valor es, como con cualquier hiperparámetro, cuestión de experimentar y comparar sobre un conjunto de validación.
